In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install netCDF4

import gzip
import shutil
import xarray as xr
import pandas as pd
import numpy as np
from geopy.distance import geodesic

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#הערה: כדי להימנע מזמן ריצה ארוך הוגבלו מספר משתנים: נלקחו רק 10 רגעי זמן, ורק 5 תחנות גשם. בפועל יש הרבה יותר.
#הערה2: כאן מטפלים בחודש אחד (קובץ אחד של גשמים ואחד תואם של הלינקים).
# כדי לעבור על כל הדאטא הקיים ניתן לבצע לולאה שמטפלת באופן זהה גם בשאר הקבצים.
#הקובץ יוצר 2 קבצי CSV:
# אחד שמציג מידע נרחב לגבי כל מדידה,
# ואחד שמציג זוגות של הנחתה בסאב לינק כלשהו, מול מדידת גשם מתאימה.



# פונקציה לחילוץ קובץ .nc.gz
def decompress_nc_gz(gz_file_path, output_nc_file_path):
    with gzip.open(gz_file_path, 'rb') as f_in:
        with open(output_nc_file_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print(f"File decompressed and saved as {output_nc_file_path}")

# שלב 1: חילוץ קבצים
compressed_CML_path = '/content/drive/MyDrive/CML_202101010000_202101312359.nc.gz'
decompressed_CML_path = '/content/CML_202101010000_202101312359.nc'
compressed_gauges_path = '/content/drive/MyDrive/AWS_202101.nc.gz'
decompressed_gauges_path = '/content/AWS_202101.nc'

# חילוץ קובץ CML
decompress_nc_gz(compressed_CML_path, decompressed_CML_path)

# חילוץ קובץ גשמים
decompress_nc_gz(compressed_gauges_path, decompressed_gauges_path)

# שלב 2: טענת קבצים עם xarray
ds_cml = xr.open_dataset(decompressed_CML_path, decode_cf=False)
ds_gauges = xr.open_dataset(decompressed_gauges_path, decode_cf=False)

# שלב 3: חילוץ נתוני זמן מהקבצים
rain_times = ds_gauges['time'].values  # זמני מדידות הגשם
signal_times = ds_cml['time'].values  # זמני האותות

# מציאת הזמנים החופפים
common_times = np.intersect1d(rain_times, signal_times)

# הגבלה על מספר דגימות הזמן (לדוגמה, 10 דגימות ראשונות)
num_times_to_process = 10
common_times = common_times[:num_times_to_process]

# יצירת רשימות לשמירת התוצאות
result = []
pairs = []

# שלב 4: חישוב תוצאות
for time in common_times:
    for cml_id in ds_cml['cml_id'].values:
        rsl_ch1 = ds_cml['rsl'].sel(cml_id=cml_id, time=time).values[0]  # RSL עבור CHANNEL 1
        tsl_ch1 = ds_cml['tsl'].sel(cml_id=cml_id, time=time).values[0]  # TSL עבור CHANNEL 1
        rsl_ch2 = ds_cml['rsl'].sel(cml_id=cml_id, time=time).values[1]  # RSL עבור CHANNEL 2
        tsl_ch2 = ds_cml['tsl'].sel(cml_id=cml_id, time=time).values[1]  # TSL עבור CHANNEL 2

        # חישוב ההנחתה עבור CHANNEL 1 ו-CHANNEL 2
        attenuation_ch1 = rsl_ch1 - tsl_ch1
        attenuation_ch2 = rsl_ch2 - tsl_ch2

        # הוצאת הקואורדינטות של אתר 0 ואתר 1
        cml_lat_0 = ds_cml['site_0_lat'].sel(cml_id=cml_id).values
        cml_lon_0 = ds_cml['site_0_lon'].sel(cml_id=cml_id).values
        cml_lat_1 = ds_cml['site_1_lat'].sel(cml_id=cml_id).values
        cml_lon_1 = ds_cml['site_1_lon'].sel(cml_id=cml_id).values

        # חישוב ממוצע הקואורדינטות (מרכז הלינק)
        cml_lat = (cml_lat_0 + cml_lat_1) / 2
        cml_lon = (cml_lon_0 + cml_lon_1) / 2

        # חיפוש תחנת הגשם הקרובה ביותר
        closest_rain_station = None
        min_distance = float('inf')
        rain_lat = None
        rain_lon = None
        rain_amount = None

        for idx in range(5):  #נסתכל רק על 5 תחנות.
        #כדי לכלול את כל התחנות נכתוב:  for idx in range(len(ds_gauges['id'])):
            rain_id = ds_gauges['id'][idx].item()  # שליפת ה-ID של תחנת הגשם
            #מציאת המיקום של התחנה:
            rain_lat = ds_gauges['latitude'].sel(id=rain_id).values
            rain_lon = ds_gauges['longitude'].sel(id=rain_id).values

            # חישוב המרחק בין תחנת הגשם למרכז הלינק
            distance = geodesic((cml_lat, cml_lon), (rain_lat, rain_lon)).kilometers
            if distance < min_distance:
                min_distance = distance
                closest_rain_station = rain_id
                rain_amount = ds_gauges['rainfall_amount'].sel(time=time, id=rain_id).values
        #בסוף לולאה זו, הנתונים הם עבור תחנת הגשם הקרובה ביותר למרכז הלינק.

        # שמירת המידע למערך התוצאות המפורט
        result.append({
            'time': time,
            'cml_id': cml_id,
            'attenuation_ch1': attenuation_ch1,  # ההנחתה עבור CHANNEL 1
            'attenuation_ch2': attenuation_ch2,  # ההנחתה עבור CHANNEL 2
            'rain_station': closest_rain_station,
            'rain_amount': rain_amount
        })

        # שמירת המידע למערך הזוגות של הנחתה-כמות גשם
        pairs.append({
            'attenuation': attenuation_ch1,  # ההנחתה עבור CHANNEL 1
            'rain_amount': rain_amount
        })
        pairs.append({
            'attenuation': attenuation_ch2,  # ההנחתה עבור CHANNEL 2
            'rain_amount': rain_amount
        })


# המרת הרשימה לדאטה-פריים (לניתוח נוח יותר)
df1 = pd.DataFrame(result)
df2 = pd.DataFrame(pairs)

#שמירה כCSV
# df1.to_csv('/content/drive/MyDrive/cml_rain_analysis_202101.csv', index=False)
# df2.to_csv('/content/drive/MyDrive/pairs_attenuation_rain_202101.csv', index=False)

# הצגת התוצאה
# print(df1.head(20))  # מציג את 20 השורות הראשונות
print(df2.head(20))

File decompressed and saved as /content/CML_202101010000_202101312359.nc
File decompressed and saved as /content/AWS_202101.nc
    attenuation  rain_amount
0      -43.0000          0.0
1      -45.0000          0.0
2      -56.5000          0.0
3      -55.5417          0.0
4      -53.0000          0.0
5      -51.0000          0.0
6      -53.0000  0.200000003
7      -52.0000  0.200000003
8      -62.0000  0.200000003
9      -62.0000  0.200000003
10     -59.0000          0.0
11     -59.0000          0.0
12     -62.0000  0.200000003
13     -61.0000  0.200000003
14     -44.0000          0.0
15     -49.0000          0.0
16     -56.0000          0.0
17     -56.0000          0.0
18     -68.0000          0.0
19     -69.0000          0.0


creating metadata as a csv file


In [ ]:
import netCDF4
import pandas as pd
import xarray as xr

# Function to list all variables in a NetCDF file
def list_nc_variables(nc_file_path):
    try:
        # Open the NetCDF file
        dataset = netCDF4.Dataset(nc_file_path, 'r')

        # Get all variable names
        variables = list(dataset.variables.keys())

        # Print the variable names
        print(f"Variables in {nc_file_path}:")
        for var in variables:
            print(f"- {var}")

        # Optionally, close the dataset
        dataset.close()
    except Exception as e:
        print(f"Error reading {nc_file_path}: {e}")

# Function to collect metadata for variables in a NetCDF file
def collect_metadata(nc_file):
    dataset = netCDF4.Dataset(nc_file, 'r')
    metadata = []
    for var_name in dataset.variables:
        var = dataset.variables[var_name]
        var_info = {
            'Variable': var_name,
            'Dimensions': var.dimensions,
            'Shape': var.shape,
            'Units': getattr(var, 'units', 'N/A'),
            'Description': getattr(var, 'description', 'N/A')
        }
        metadata.append(var_info)
    dataset.close()
    return metadata

In [ ]:
list_nc_variables(decompressed_CML_path)
list_nc_variables(decompressed_gauges_path)

cml_metadata = collect_metadata(decompressed_CML_path)
cml_metadata_df = pd.DataFrame(cml_metadata)
gauges_metadata = collect_metadata(decompressed_gauges_path)
gauges_metadata_df = pd.DataFrame(gauges_metadata)

print(cml_metadata_df)
print(gauges_metadata_df)



Variables in /content/CML_202101010000_202101312359.nc:
- time
- sublink_id
- cml_id
- rsl
- tsl
- length
- site_0_lat
- site_0_lon
- site_0_elev
- site_1_lat
- site_1_lon
- site_1_elev
- frequency
- polarization
Variables in /content/AWS_202101.nc:
- time
- id
- rainfall_amount
- temperature
- relative_humidity
- wind_velocity
- wind_direction
- latitude
- longitude
- elevation
        Variable                  Dimensions            Shape  \
0           time                     (time,)         (44640,)   
1     sublink_id               (sublink_id,)             (2,)   
2         cml_id                   (cml_id,)           (151,)   
3            rsl  (cml_id, sublink_id, time)  (151, 2, 44640)   
4            tsl  (cml_id, sublink_id, time)  (151, 2, 44640)   
5         length                   (cml_id,)           (151,)   
6     site_0_lat                   (cml_id,)           (151,)   
7     site_0_lon                   (cml_id,)           (151,)   
8    site_0_elev                 

In [ ]:
dataset = xr.open_dataset(decompressed_CML_path)

# Limit the data to reduce memory usage
time_subset = dataset['time'].values[:1000]  # Limit to the first 1000 time points
cml_id_subset = dataset['cml_id'].values[:10]  # Limit to the first 10 cml_id values

# Convert time to pandas datetime
time = pd.to_datetime(time_subset, unit='s')

# Initialize an empty list to store data incrementally
data = []

# Process data in chunks
for i, cid in enumerate(cml_id_subset):
    rsl_chunk = dataset['rsl'].sel(cml_id=cid).values[:, :1000]  # Subset along time
    tsl_chunk = dataset['tsl'].sel(cml_id=cid).values[:, :1000]  # Subset along time
    length = dataset['length'].sel(cml_id=cid).values
    site_0_lat = dataset['site_0_lat'].sel(cml_id=cid).values
    site_0_lon = dataset['site_0_lon'].sel(cml_id=cid).values
    site_0_elev = dataset['site_0_elev'].sel(cml_id=cid).values
    site_1_lat = dataset['site_1_lat'].sel(cml_id=cid).values
    site_1_lon = dataset['site_1_lon'].sel(cml_id=cid).values
    site_1_elev = dataset['site_1_elev'].sel(cml_id=cid).values
    frequency = dataset['frequency'].sel(cml_id=cid).values
    polarization = dataset['polarization'].sel(cml_id=cid).values

    for j, sid in enumerate(dataset['sublink_id'].values):
        for k, t in enumerate(time):
            data.append([
                t, cid, sid,
                rsl_chunk[j, k], tsl_chunk[j, k], length,
                site_0_lat, site_0_lon, site_0_elev,
                site_1_lat, site_1_lon, site_1_elev,
                frequency[j], polarization[j]
            ])

        # Periodically save to avoid keeping everything in memory
        if len(data) > 100000:  # Save every 100,000 rows
            df_chunk = pd.DataFrame(data, columns=[
                'time', 'cml_id', 'sublink_id', 'rsl', 'tsl', 'length',
                'site_0_lat', 'site_0_lon', 'site_0_elev',
                'site_1_lat', 'site_1_lon', 'site_1_elev',
                'frequency', 'polarization'
            ])
            df_chunk.to_csv('output_data.csv', mode='a', header=False, index=False)
            data = []  # Clear the list to free memory

# Save remaining data
if data:
    df = pd.DataFrame(data, columns=[
        'time', 'cml_id', 'sublink_id', 'rsl', 'tsl', 'length',
        'site_0_lat', 'site_0_lon', 'site_0_elev',
        'site_1_lat', 'site_1_lon', 'site_1_elev',
        'frequency', 'polarization'
    ])
    df.to_csv('output_data.csv', mode='a', header=False, index=False)

print("Data saved successfully!")






Data saved successfully!
